<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone "https://github.com/slomi23/ML_fx.git"
!cd ML_fx/

Cloning into 'ML_fx'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 36 (delta 5), reused 18 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 8.95 MiB | 12.73 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [7]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

PROCCESSED_DATA_DIR = "./ML_fx/data/processed/"
train_full=pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, "train_processed.csv"))
print(train_full.head())


   Store  Dept        Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1  2010-02-05      24924.50      False        42.31       2.572   
1      1     1  2010-02-12      46039.49       True        38.51       2.548   
2      1     1  2010-02-19      41595.55      False        39.93       2.514   
3      1     1  2010-02-26      19403.54      False        46.63       2.561   
4      1     1  2010-03-05      21827.90      False        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  MarkDown4  MarkDown5         CPI  \
0        NaN        NaN        NaN        NaN        NaN  211.096358   
1        NaN        NaN        NaN        NaN        NaN  211.242170   
2        NaN        NaN        NaN        NaN        NaN  211.289143   
3        NaN        NaN        NaN        NaN        NaN  211.319643   
4        NaN        NaN        NaN        NaN        NaN  211.350143   

   Unemployment Type    Size  
0         8.106    A  151315  
1         8.106    A  15

# Feature Engineering
let's create sales_lag52, sales  year ago. as well as some calendary features

In [10]:
def create_features(df1):
  df=df1.copy()
  group_cols = ['Store', 'Dept']
  df['Date'] = pd.to_datetime(df['Date'])

  df['sales_lag_52'] = df.groupby(group_cols)['Weekly_Sales'].shift(52)
  df['Year'] = df['Date'].dt.year
  df['Month'] = df['Date'].dt.month
  df['DayOfWeek'] = df['Date'].dt.dayofweek
  df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)

  #sine and cosine
  df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
  df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)
  df['dow_sin'] = np.sin(2 * np.pi * df['DayOfWeek']/7)
  df['dow_cos'] = np.cos(2 * np.pi * df['DayOfWeek']/7)
  df['week_sin'] = np.sin(2 * np.pi * df['WeekOfYear']/52)
  df['week_cos'] = np.cos(2 * np.pi * df['WeekOfYear']/52)
  #drop columns not needed
  cols_to_drop = ['Month', 'DayOfWeek', 'WeekOfYear', 'Date']
  df=df.drop(columns=cols_to_drop)
  return df
train_full1=create_features(train_full)
print(train_full1.head())


   Store  Dept  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  MarkDown1  \
0      1     1      24924.50      False        42.31       2.572        NaN   
1      1     1      46039.49       True        38.51       2.548        NaN   
2      1     1      41595.55      False        39.93       2.514        NaN   
3      1     1      19403.54      False        46.63       2.561        NaN   
4      1     1      21827.90      False        46.50       2.625        NaN   

   MarkDown2  MarkDown3  MarkDown4  ...  Type    Size  sales_lag_52  Year  \
0        NaN        NaN        NaN  ...     A  151315           NaN  2010   
1        NaN        NaN        NaN  ...     A  151315           NaN  2010   
2        NaN        NaN        NaN  ...     A  151315           NaN  2010   
3        NaN        NaN        NaN  ...     A  151315           NaN  2010   
4        NaN        NaN        NaN  ...     A  151315           NaN  2010   

   month_sin     month_cos   dow_sin   dow_cos  week_sin  week

# Now let's handle NAs/Nulls and turn categorical features into numerical
target encoding for Type

In [12]:
numerical_cols = train_full1.select_dtypes(include=['number']).columns
print(numerical_cols)

Index(['Store', 'Dept', 'Weekly_Sales', 'Temperature', 'Fuel_Price',
       'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI',
       'Unemployment', 'Size', 'sales_lag_52', 'Year', 'month_sin',
       'month_cos', 'dow_sin', 'dow_cos', 'week_sin', 'week_cos'],
      dtype='object')


In [13]:
def fill_missing_values(df1):
  df=df1.copy()
  numerical_cols = df.select_dtypes(include=['number']).columns
  df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns
  for col in categorical_cols:
      mode_val = df[col].mode() if not df[col].mode().empty else None
      if mode_val is not None:
          df[col] = df[col].fillna(mode_val)
  return df
train_full2=fill_missing_values(train_full1)
print(train_full2.head())


   Store  Dept  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  MarkDown1  \
0      1     1      24924.50      False        42.31       2.572    5347.45   
1      1     1      46039.49       True        38.51       2.548    5347.45   
2      1     1      41595.55      False        39.93       2.514    5347.45   
3      1     1      19403.54      False        46.63       2.561    5347.45   
4      1     1      21827.90      False        46.50       2.625    5347.45   

   MarkDown2  MarkDown3  MarkDown4  ...  Type    Size  sales_lag_52  Year  \
0      192.0       24.6    1481.31  ...     A  151315       7998.55  2010   
1      192.0       24.6    1481.31  ...     A  151315       7998.55  2010   
2      192.0       24.6    1481.31  ...     A  151315       7998.55  2010   
3      192.0       24.6    1481.31  ...     A  151315       7998.55  2010   
4      192.0       24.6    1481.31  ...     A  151315       7998.55  2010   

   month_sin     month_cos   dow_sin   dow_cos  week_sin  week

In [15]:
def cat_to_num(df1):
  df=df1.copy()
  df['IsHoliday'] = df['IsHoliday'].astype(int)
  type_map = {'A': 20, 'B': 12, 'C': 9.5}
  if 'Type' in df.columns:
      df['Type'] = df['Type'].map(type_map).fillna(0).astype(int)
  return df
train_full3=cat_to_num(train_full2)
print(train_full3.head())

   Store  Dept  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  MarkDown1  \
0      1     1      24924.50          0        42.31       2.572    5347.45   
1      1     1      46039.49          1        38.51       2.548    5347.45   
2      1     1      41595.55          0        39.93       2.514    5347.45   
3      1     1      19403.54          0        46.63       2.561    5347.45   
4      1     1      21827.90          0        46.50       2.625    5347.45   

   MarkDown2  MarkDown3  MarkDown4  ...  Type    Size  sales_lag_52  Year  \
0      192.0       24.6    1481.31  ...    20  151315       7998.55  2010   
1      192.0       24.6    1481.31  ...    20  151315       7998.55  2010   
2      192.0       24.6    1481.31  ...    20  151315       7998.55  2010   
3      192.0       24.6    1481.31  ...    20  151315       7998.55  2010   
4      192.0       24.6    1481.31  ...    20  151315       7998.55  2010   

   month_sin     month_cos   dow_sin   dow_cos  week_sin  week